**Tests  statistiques**
-----------------------

In [2]:
import numpy as np 
import pandas as pd 
import seaborn as sns
from scipy import stats

In [3]:
# Chargement des données : 
df_val= pd.read_csv("/Users/thomasalaby/Desktop/ML-PRO/ML_PROJECT/flight_price_prediction/data/processed/validation.csv")
df_val.head(5)

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,31941,Air_India,AI-665,Delhi,Early_Morning,one,Night,Hyderabad,Economy,13.58,26,4209
1,144478,Vistara,UK-772,Kolkata,Morning,two_or_more,Early_Morning,Hyderabad,Economy,21.42,19,7834
2,230195,Vistara,UK-960,Mumbai,Morning,one,Morning,Bangalore,Business,20.75,4,66928
3,104641,AirAsia,I5-1562,Bangalore,Afternoon,one,Night,Kolkata,Economy,7.50,16,3393
4,101621,Vistara,UK-810,Bangalore,Early_Morning,one,Evening,Mumbai,Economy,9.50,43,6568


In [4]:
df_val.info()

<class 'pandas.DataFrame'>
RangeIndex: 45023 entries, 0 to 45022
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        45023 non-null  int64  
 1   airline           45023 non-null  str    
 2   flight            45023 non-null  str    
 3   source_city       45023 non-null  str    
 4   departure_time    45023 non-null  str    
 5   stops             45023 non-null  str    
 6   arrival_time      45023 non-null  str    
 7   destination_city  45023 non-null  str    
 8   class             45023 non-null  str    
 9   duration          45023 non-null  float64
 10  days_left         45023 non-null  int64  
 11  price             45023 non-null  int64  
dtypes: float64(1), int64(3), str(8)
memory usage: 6.4 MB


Le seuil de signification choisi pour ce test est de 5% ($\alpha=0.05$)

In [5]:
alpha=0.05

À partir des graphiques et des statistiques calculées précédemment, on retient les observations suivantes :

1.**duration vs prix**.  
**Question**: les vols plus longs sont-ils associés à des prix plus élevés?
* `duration`(variable continue)

* `price`: (variable continue)

**Hypothèses**:
* $H_0$: la corrélation linéaire entre `duration`et `price`est nulle.
* $H1:$ la corrélation linéaire entre `duration` et `price` est non nulle

In [6]:
# Test de corrélation entre duration et price
corr_duration_price, p_value_duration_price = stats.spearmanr(df_val["price"],df_val["duration"])

print("Corrélation (duration, price) :", corr_duration_price)
print("p-value :", p_value_duration_price)

if p_value_duration_price < alpha:
    print(f"→ p-value < {alpha} : on rejette H0.")
    print("Conclusion : il existe une corrélation linéaire significative entre la durée du vol et le prix.")
else:
    print(f"→ p-value ≥ {alpha} : on ne rejette pas H0.")
    print("Conclusion : on ne met pas en évidence de corrélation linéaire significative entre la durée du vol et le prix.")

Corrélation (duration, price) : 0.3213008609135181
p-value : 0.0
→ p-value < 0.05 : on rejette H0.
Conclusion : il existe une corrélation linéaire significative entre la durée du vol et le prix.


2.**Airline vs Price**. 

**Question**: Les différentes compagnies pratiquent-elles en moyenne le même le prix  ?
* `Àirline` : variable discrète (plusieure catégorie)
* `price`: (variable continue). 

**Hypothése**: 
* $H_{0}$: le prix moyen des billets est le même pour tous les compagnies considérées. 
* $H_{1}$: au moins une compagnies a un prix moyen différent des autres 


In [7]:
# Test de liaisons entre duration et price
groups = [
    group["price"].dropna()
    for _, group in df_val.groupby("airline")
]

statistic, p_value = stats.kruskal(*groups)

print("Statistique H :", statistic)
print("p-value :", p_value)

if p_value < alpha:
    print(f"p-value < {alpha} : on rejette H0.")
    print("conclusion: le prix moyen n'est pas le même pour toutes ces compagnies(au moins une se distingue).")
else:
    print(f"p-value ≥ {alpha} : on ne rejette pas H0.")
    print("Aucune différence significative de prix n’a été mise en évidence.")

Statistique H : 14141.157059681813
p-value : 0.0
p-value < 0.05 : on rejette H0.
conclusion: le prix moyen n'est pas le même pour toutes ces compagnies(au moins une se distingue).


3.**departure_time vs price**.   
**Question** le prix moyen des billets varie t-il en fonction de l'heure de depart ?  
  * `departure_time` : variable discrète(plusieurs catégories ordinales).  
  * `price` : variable continue. 
    
**Hypothèse**
  * $H_0$: le prix moyen des billets est le même pour tous les horraires de départs 
  * $H_1$: au moins une des horraires présente un prix différent

In [8]:
# Test de liaisons entre departure_time et price
groups = [
    group["price"].dropna()
    for _, group in df_val.groupby("departure_time")
]

statistic, p_value = stats.kruskal(*groups)

print("Statistique H :", statistic)
print("p-value :", p_value)

if p_value < alpha:
    print(f"p-value < {alpha} : on rejette H0.")
    print("conclusion: le prix moyen n'est pas le même pour toutes ces horraire(au moins une se distingue).")
else:
    print(f"p-value ≥ {alpha} : on ne rejette pas H0.")
    print("Aucune différence significative de prix n’a été mise en évidence.")

Statistique H : 349.45362968664585
p-value : 2.2947458504336125e-73
p-value < 0.05 : on rejette H0.
conclusion: le prix moyen n'est pas le même pour toutes ces horraire(au moins une se distingue).


4.**stops vs price**.  
**Question** le prix moyen est il le même pour tous les escales ?  
 * `stops`: variable discrète(ordinale)
 * `price`: variables continue. 

**Hypothèse**: 
 * $H_0$: Le prix moyen des billets est le même pour tous les escales 
 * $H_1$: au moins un des escales affiche un prix moyens differents

In [9]:
# Test de liaisons entre stops et price
groups = [
    group["price"].dropna()
    for _, group in df_val.groupby("stops")
]

statistic, p_value = stats.kruskal(*groups)

print("Statistique H :", statistic)
print("p-value :", p_value)

if p_value < alpha:
    print(f"p-value < {alpha} : on rejette H0.")
    print("conclusion: le prix moyen n'est pas le même pour toutes les escales(au moins une se distingue).")
else:
    print(f"p-value ≥ {alpha} : on ne rejette pas H0.")
    print("Aucune différence significative de prix n’a été mise en évidence.")

Statistique H : 3107.9051397924954
p-value : 0.0
p-value < 0.05 : on rejette H0.
conclusion: le prix moyen n'est pas le même pour toutes les escales(au moins une se distingue).


5. **days_left vs price**.  
**Questions** : Le prix moyen varie t-il en fonction de l'approche de depart ?
 * `days_left`: variable continue
 * `price`: variable continue

 **Hypothèse**:  
 * $H_0$: la correllation entre les variables days_left et price est **nulle**
 * $H_1$: la correllation entre les variables days_left et price est **non nulle**


In [10]:
# Test de corrélation entre duration et price
corr_duration_price, p_value_duration_price = stats.spearmanr(df_val["price"],df_val["days_left"])

print("Corrélation (duration, price) :", corr_duration_price)
print("p-value :", p_value_duration_price)

if p_value_duration_price < alpha:
    print(f"→ p-value < {alpha} : on rejette H0.")
    print("Conclusion : il existe une corrélation linéaire significative entre jour de depart et le prix.")
else:
    print(f"→ p-value ≥ {alpha} : on ne rejette pas H0.")
    print("Conclusion : on ne met pas en évidence de corrélation linéaire significative entre la durée du vol et le prix.")

Corrélation (duration, price) : -0.26850961899569936
p-value : 0.0
→ p-value < 0.05 : on rejette H0.
Conclusion : il existe une corrélation linéaire significative entre jour de depart et le prix.
